# S01. Multi-World Systems

A `System` holds several worlds and the orbits that connect them. Each orbiting world has a tidal host (the body that raises its tides) and a semi-major axis and eccentricity about that host. One world is the star, which provides insolation and is often also the tidal host. The system calculates every world's instantaneous orbital and spin evolution in one call.

This notebook builds a small planetary system, calculates its evolution, adds a moon with its planet as its tidal host, and saves, reloads, and trims the system.

In [1]:
%matplotlib inline
import shutil
import tempfile
from pathlib import Path

import numpy as np

from TidalPy.constants import au, year
from TidalPy.Structures import StarWorld, System, build_system, build_world, load_system, make_tide

GYR = 1.0e9 * year  # [s]


def tidal_planet(config, love_k2, quality_factor):
    "Build a world and give it a degree-2 fixed-Q tide so it can dissipate."
    world = build_world(config)
    world.set_tide_model(make_tide("fixed_q", fixed_k=[love_k2], fixed_q=[quality_factor]))
    world.set_tide_config(max_degree_l=2)  # The fixed-Q lists give degree 2 only
    return world


star = StarWorld("Sun", 6.957e8, 1.988e30)
star.set_effective_temperature(5772.0)

# A rocky planet with k2 = 0.3 and Q = 100
inner = tidal_planet(
    {"name": "Inner", "type": "terrestrial", "radius_m": 6.0e6, "mass_kg": 5.0e24,
     "layers": {"core": {"material": "simple_iron_core", "radius_outer_m": 3.0e6, "use_tides": False},
                "mantle": {"material": "simple_rock", "radius_fraction": 1.0}}},
    love_k2=0.3,
    quality_factor=100.0)
# A gas giant with k2 = 0.4 and Q = 1e4 (Jupiter's k2/Q is about 1.1e-5, Lainey et al. 2009)
outer = tidal_planet(
    {"name": "Outer", "type": "gasgiant", "radius_m": 6.0e7, "mass_kg": 6.0e26,
     "layers": {"envelope": {"material": "simple_gas", "radius_fraction": 1.0}}},
    love_k2=0.4,
    quality_factor=1.0e4)

system = System("Demo-System")
system.add_world(star, is_star=True)
# synchronous=True starts each planet with its spin equal to its mean motion
system.add_world(inner, tidal_host=star, semi_major_axis=0.20 * au, eccentricity=0.05, synchronous=True)
system.add_world(outer, tidal_host=star, semi_major_axis=0.80 * au, eccentricity=0.10, synchronous=True)

print(repr(system))
print("tidal hosts:", ", ".join(f"{world.name} -> {system.get_tidal_host(world).name}" for world in (inner, outer)))

System('Demo-System', worlds=['Sun', 'Inner', 'Outer'], star='Sun')
tidal hosts: Inner -> Sun, Outer -> Sun


## System Evolution

`calc_system_evolution` returns one record per world, named by its `world_name`, with its instantaneous tidal heating and orbital and spin rates. The Sun has no tidal host, so its record has `evolved = False` and zero rates. Both planets start in synchronous rotation on eccentric orbits, so `dspin/dt` is positive: the tide spins them up toward the faster pseudo-synchronous rate of an eccentric orbit. The last two columns turn the rates into timescales, $a / |\dot{a}|$ and $e / |\dot{e}|$. Both are far longer than the age of the Solar System, as expected for planets this far from their star.

In [2]:
print(f"{'world':6} {'heating (W)':>12} {'da/dt (m/s)':>12} {'de/dt (1/s)':>12} {'dspin/dt (rad/s^2)':>19} "
      f"{'a/|da/dt| (Gyr)':>16} {'e/|de/dt| (Gyr)':>16}")
for row in system.calc_system_evolution():
    if not row["evolved"]:
        print(f"{row['world_name']:6} not evolved (no tidal host)")
        continue
    semi_major_axis_timescale = row["semi_major_axis"] / abs(row["da_dt"]) / GYR
    eccentricity_timescale = row["eccentricity"] / abs(row["de_dt"]) / GYR
    print(f"{row['world_name']:6} {row['tidal_heating']:12.3e} {row['da_dt']:12.3e} {row['de_dt']:12.3e} "
          f"{row['dspin_dt']:19.3e} {semi_major_axis_timescale:16.2e} {eccentricity_timescale:16.2e}")

world   heating (W)  da/dt (m/s)  de/dt (1/s)  dspin/dt (rad/s^2)  a/|da/dt| (Gyr)  e/|de/dt| (Gyr)
Sun    not evolved (no tidal host)
Inner     5.103e+11   -3.705e-12   -4.581e-22           5.382e-21         2.56e+05         3.46e+03
Outer     8.770e+10   -8.268e-14   -1.294e-24           5.910e-25         4.59e+07         2.45e+06


## Reading and Setting Orbital Elements

Orbital elements can be read and changed at any time. Tidal heating grows with eccentricity, so raising it immediately increases that world's dissipation. `calc_world_evolution` returns the record of one world.

In [3]:
heating_at_start = system.calc_world_evolution(inner)["tidal_heating"]  # [W]
print("Inner's eccentricity:", system.get_eccentricity(inner), end=" -> ")
system.set_eccentricity(inner, 0.10)
print(system.get_eccentricity(inner))

heating_at_new_eccentricity = system.calc_world_evolution(inner)["tidal_heating"]  # [W]
print(f"Inner heating at e = 0.10: {heating_at_new_eccentricity:.3e} W, "
      f"{heating_at_new_eccentricity / heating_at_start:.1f} times its value at e = 0.05")

Inner's eccentricity: 0.05 -> 0.1
Inner heating at e = 0.10: 2.155e+12 W, 4.2 times its value at e = 0.05


## Adding a Moon

`add_world` adds a world at any time, and `set_tidal_host` names or changes a world's tidal host. A moon's tides are raised by its planet, not by the star, so its tidal orbit is about the planet. Its orbit about the star, which sets its insolation, is given separately.

Below, a moon is added with no tidal host, then made Outer's satellite with `set_tidal_host` and given its two orbits, and `set_synchronous_rotation` sets its spin to its new mean motion. `add_world` can do the same in one call, with `tidal_host`, `semi_major_axis`, `eccentricity`, `synchronous=True`, `stellar_semi_major_axis`, and `stellar_eccentricity`. `set_star` moves the star role in the same way.

In [4]:
moon = tidal_planet(
    {"name": "Moon", "type": "terrestrial", "radius_m": 1.8e6, "mass_kg": 8.0e22,
     "layers": {"mantle": {"material": "simple_rock", "radius_fraction": 1.0}}},
    love_k2=0.03,
    quality_factor=100.0)
system.add_world(moon)
print("Moon's tidal host after add_world:", system.get_tidal_host(moon))

system.set_tidal_host(moon, outer)  # Outer raises the Moon's tides
system.set_semi_major_axis(moon, 1.0e9)  # Orbit about Outer [m]
system.set_eccentricity(moon, 0.01)
system.set_stellar_semi_major_axis(moon, 0.80 * au)  # Orbit about the Sun, for insolation [m]
system.set_stellar_eccentricity(moon, 0.10)
moon_spin = system.set_synchronous_rotation(moon)  # [rad s-1]

moon_evolution = system.calc_world_evolution(moon)
moon_period = 2.0 * np.pi / moon_spin / 86400.0  # [days]
print(f"Moon's tidal host: {system.get_tidal_host(moon).name}, orbital period {moon_period:.2f} days")
print(f"Moon tidal heating: {moon_evolution['tidal_heating']:.3e} W, "
      f"insolation: {system.calc_insolation_flux(moon):.0f} W/m^2")
print(f"now {system.num_worlds} worlds: {[world.name for world in system]}")

Moon's tidal host after add_world: None
Moon's tidal host: Outer, orbital period 11.49 days
Moon tidal heating: 9.057e+08 W, insolation: 2138 W/m^2
now 4 worlds: ['Sun', 'Inner', 'Outer', 'Moon']


## Saving, Loading, and Trimming the System

A system saves to TOML, a readable recipe, or to a binary snapshot, as in the Basics notebook. The TOML file holds the system's orbits and roles in one `[worlds.<name>]` table per world, followed by each world's own configuration. `build_system` rebuilds a system from the file, and `load_system` from the binary snapshot, which restores each world as its concrete type.

There is no in-place remove. To drop a world, rebuild the system from its configuration (`get_save_config`, the dictionary the TOML file is written from) without that world.

In [5]:
work = Path(tempfile.mkdtemp(prefix="tidalpy_sys_"))

# TOML round trip
toml_path = work / "system.toml"
system.save_to_toml(toml_path)
toml_text = toml_path.read_text()
print(toml_text[toml_text.index("[worlds.Sun]"):toml_text.index("[worlds.Sun.world]")].strip())  # Orbits and roles
from_toml = build_system(toml_path)
print()
print(f"{'world':6} {'heating (W)':>12} {'from TOML (W)':>14}")
for world in (inner, outer, moon):
    original = system.calc_world_evolution(world.name)["tidal_heating"]
    rebuilt = from_toml.calc_world_evolution(world.name)["tidal_heating"]
    print(f"{world.name:6} {original:12.4e} {rebuilt:14.4e}")

# Binary round trip
system.save_binary(work / "system.tpyb")
from_binary = load_system(work / "system.tpyb")
print()
print("from binary:", [(world.name, type(world).__name__) for world in from_binary])

# Drop the moon by rebuilding the system without it
config = system.get_save_config()
del config["worlds"]["Moon"]
without_moon = build_system(config)
print("without the moon:", [world.name for world in without_moon])

shutil.rmtree(work, ignore_errors=True)

[worlds.Sun]
is_star = true

[worlds.Inner]
tidal_host = "Sun"
semi_major_axis_m = 29919574140.0
eccentricity = 0.1

[worlds.Outer]
tidal_host = "Sun"
semi_major_axis_m = 119678296560.0
eccentricity = 0.1

[worlds.Moon]
tidal_host = "Outer"
semi_major_axis_m = 1000000000.0
eccentricity = 0.01
stellar_semi_major_axis_m = 119678296560.0
stellar_eccentricity = 0.1

world   heating (W)  from TOML (W)
Inner    2.1549e+12     2.1549e+12
Outer    8.7697e+10     8.7697e+10
Moon     9.0574e+08     9.0574e+08

from binary: [('Sun', 'StarWorld'), ('Inner', 'TerrestrialWorld'), ('Outer', 'GasGiantWorld'), ('Moon', 'TerrestrialWorld')]
without the moon: ['Sun', 'Inner', 'Outer']
